// Copyright (C) 2026 Naver Corporation. All rights reserved.


The following scripts assume that results are laid out in paths with the following structure: `output_path/model_name/results.parquet`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
models = [
    # FILL IN THE MODEL NAMES
]

sft_model = [m for m in models if 'SFT' in m]
amari_models = [m for m in models if 'alpha' in m]
grpo_models = [m for m in models if 'dr_grpo' in m]
other_models = [m for m in models if m not in grpo_models and m not in amari_models and m not in sft_model]

amari_colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(amari_models)))
grpo_colors = plt.cm.Blues(np.linspace(0.9, 0.9, len(grpo_models)))
other_colors = plt.cm.Greens(np.linspace(0.5, 0.9, len(other_models)))
markers = ['o', 's', 'v', '^', '<', '>', 'D', 'p', 'X']

model_color_map = {}
model_marker_map = {}

for model, color in zip(amari_models, amari_colors): model_color_map[model] = color
for model, color in zip(grpo_models, grpo_colors): model_color_map[model] = color
for model, color in zip(other_models, other_colors): model_color_map[model] = color
if sft_model: model_color_map[sft_model[0]] = 'grey'
for i, model in enumerate(models): model_marker_map[model] = markers[i % len(markers)]

In [ ]:
%load_ext autoreload
%autoreload 2
from generate_plots_config import model_name_map
import pandas as pd
import math
import matplotlib.pyplot as plt
import os
import numpy as np
import json
from matplotlib.collections import PathCollection
from matplotlib.lines import Line2D
from adjustText import adjust_text
from scipy.spatial import ConvexHull
from scipy.special import comb as scipy_comb
from scipy.interpolate import interp1d

def combine_path_collections(pathcollections: list[PathCollection]) -> PathCollection:
    all_paths, all_sizes, all_offsets, all_facecolors, all_edgecolors = [], [], [], [], []
    for pc in pathcollections:
        if pc.get_paths():
            all_paths.extend(pc.get_paths() * len(pc.get_offsets()))
            all_sizes.extend(pc.get_sizes())
            all_offsets.extend(pc.get_offsets())
            all_facecolors.extend(pc.get_facecolors())
            all_edgecolors.extend(pc.get_edgecolors())
    combined_pc = PathCollection(all_paths, sizes=all_sizes, facecolors=all_facecolors, edgecolors=all_edgecolors)
    combined_pc.set_offsets(np.array(all_offsets))
    return combined_pc

def pass_at_k(n, c, k):
    if c == 0: return 0.0
    if n - c < k: return 1.0
    if n > 1000:
        return 1.0 - scipy_comb(n - c, k, exact=False) / scipy_comb(n, k, exact=False)
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)

def bootstrap_pass_at_k_std(results_series, k, n_resamples=100):
    bootstrap_stats = []
    for _ in range(n_resamples):
        boot_sample = results_series.sample(n=len(results_series), replace=True)
        pass_k_stat = boot_sample.apply(lambda x: pass_at_k(len(x), int(sum(x)), k)).mean()
        bootstrap_stats.append(pass_k_stat)
    return np.std(bootstrap_stats)

def bootstrap_pass_at_k_std_from_accuracies(accuracies, k, n_samples_json, n_resamples=100):
    bootstrap_stats = []
    accuracies_np = np.array(accuracies)
    for _ in range(n_resamples):
        boot_sample = np.random.choice(accuracies_np, size=len(accuracies_np), replace=True)
        pass_k_vals = [pass_at_k(n_samples_json, int(round(acc * n_samples_json)), k) for acc in boot_sample]
        bootstrap_stats.append(np.mean(pass_k_vals))
    return np.std(bootstrap_stats)

def get_convex_hull_frontier(points):
    """Computes the upper-right (Pareto) frontier of the convex hull."""
    if len(points) < 2:
        return points
    if len(points) < 3:
        return points[np.argsort(points[:, 0])]
    
    hull = ConvexHull(points)
    hull_points = points[hull.vertices]
    hull_points = hull_points[np.argsort(hull_points[:, 0])]
    
    frontier = []
    max_y = -1.0
    for pt in reversed(hull_points):
        if pt[1] > max_y:
            frontier.append(pt)
            max_y = pt[1]
    return np.array(sorted(frontier, key=lambda x: x[0]))

def plot_convex_hull_comparison(models, model_name_map, paths, k_values=(1, 256), n_samples_json=128, use_bootstrap=True, show_errorbars=False):
    if not isinstance(paths, list): paths = [paths]
    plt.figure(figsize=(14, 8))
    k1, k2 = k_values
    plot_data = {}

    print("Searching for result files...")
    for model in models:
        mean_pass_k1, mean_pass_k2, std_pass_k1, std_pass_k2 = None, None, 0.0, 0.0
        found_data = False

        for path in paths:
            if found_data: break
            for parquet_file in [os.path.join(path, model, "results.parquet")]:
                if os.path.exists(parquet_file):
                    ds = pd.read_parquet(parquet_file)
                    results_series = ds['results']
                    mean_pass_k1 = results_series.apply(lambda x: pass_at_k(len(x), int(sum(x)), k1)).mean()
                    mean_pass_k2 = results_series.apply(lambda x: pass_at_k(len(x), int(sum(x)), k2)).mean()
                    if use_bootstrap:
                        std_pass_k1 = bootstrap_pass_at_k_std(results_series, k=k1)
                        std_pass_k2 = bootstrap_pass_at_k_std(results_series, k=k2)
                    found_data = True; break

            json_file = os.path.join(path, f"{model}.eval.json")
            if not found_data and os.path.exists(json_file):
                with open(json_file, 'r') as f:
                    data = json.load(f)
                if not data or not isinstance(list(data.values())[0], list): continue
                accuracies = list(data.values())[0]
                mean_pass_k1 = np.mean([pass_at_k(n_samples_json, int(round(acc * n_samples_json)), k1) for acc in accuracies])
                mean_pass_k2 = np.mean([pass_at_k(n_samples_json, int(round(acc * n_samples_json)), k2) for acc in accuracies])
                if use_bootstrap:
                    std_pass_k1 = bootstrap_pass_at_k_std_from_accuracies(accuracies, k1, n_samples_json)
                    std_pass_k2 = bootstrap_pass_at_k_std_from_accuracies(accuracies, k2, n_samples_json)
                found_data = True

        if found_data:
            label = model_name_map.get(model, model)
            plot_data[model] = {'label': label, 'pass_k1': mean_pass_k1, 'pass_k2': mean_pass_k2, 'std_k1': std_pass_k1, 'std_k2': std_pass_k2}

    if not plot_data: return

    markers = {'DPG': 's', 'Other': 'o'}
    texts, xs, ys, points = [], [], [], []
    
    for i, (model, data) in enumerate(plot_data.items()):
        x, y = data['pass_k1'], data['pass_k2']
        xs.append(x); ys.append(y)
        is_dpg = 'DPG' in data['label']
        color = 'C0' if is_dpg else 'C1'
        marker = markers['DPG'] if is_dpg else markers['Other']
        
        if show_errorbars and use_bootstrap:
            plt.errorbar(x, y, yerr=data['std_k2'], xerr=data['std_k1'],
                         fmt=marker, color=color, markersize=8, capsize=4, alpha=0.8, zorder=10)
        else:
            p = plt.scatter(x, y, s=100, marker=marker, c=color, alpha=0.9, zorder=10)
            points.append(p)
        
        texts.append(plt.text(x, y, data['label'], fontsize=14, zorder=11))
    
    alpha_dpg_proxy = Line2D([0], [0], marker='s', color='w', label='$\\alpha$-DPG models',
                         markerfacecolor='C0', markersize=10, linestyle='None')
    baseline_proxy = Line2D([0], [0], marker='o', color='w', label='Baseline methods',
                        markerfacecolor='C1', markersize=10, linestyle='None')

    

    if len(plot_data) >= 2:
        model_keys = list(plot_data.keys())
        all_points = np.array([(plot_data[m]['pass_k1'], plot_data[m]['pass_k2']) for m in model_keys])
        frontier_pts = get_convex_hull_frontier(all_points)
        
        if len(frontier_pts) >= 2:
            frontier_stds_x = []
            frontier_stds_y = []
            for f_pt in frontier_pts:
                idx = np.where((all_points == f_pt).all(axis=1))[0][0]
                m_key = model_keys[idx]
                frontier_stds_x.append(plot_data[m_key]['std_k1'])
                frontier_stds_y.append(plot_data[m_key]['std_k2'])

            num_layers = 2
            for l in range(num_layers):
                l_frontier_stds_x = np.array(frontier_stds_x) / num_layers * l
                l_frontier_stds_y = np.array(frontier_stds_y) / num_layers * l

                # Draw the primary Pareto line
                plt.plot(frontier_pts[:,0], frontier_pts[:,1], linestyle='--', color='black', 
                        linewidth=2, label='Pareto Frontier', zorder=5)
                
                if use_bootstrap:
                    # Core corner points
                    x_out_core = frontier_pts[:, 0] + l_frontier_stds_x
                    y_out_core = frontier_pts[:, 1] + l_frontier_stds_y
                    x_in_core = frontier_pts[:, 0] - l_frontier_stds_x
                    y_in_core = np.maximum(0, frontier_pts[:, 1] - l_frontier_stds_y)
                    
                    # Add horizontal segments at the borders
                    # Left border (start point): horizontal at y1 + sy1 from x1-sx1 to x1+sx1
                    x_out = np.concatenate([[frontier_pts[0, 0] - l_frontier_stds_x[0]], x_out_core])
                    y_out = np.concatenate([[y_out_core[0]], y_out_core])
                    
                    # Right border (end point): horizontal at yn - syn from xn-sxn to xn+sxn
                    x_in = np.concatenate([x_in_core, [frontier_pts[-1, 0] + l_frontier_stds_x[-1]]])
                    y_in = np.concatenate([y_in_core, [y_in_core[-1]]])
                    y_in = np.maximum(y_in, y_in_core[-1])

                    
                    # Global range covering the full extension
                    x_min_full = min(x_out.min(), x_in.min())
                    x_max_full = max(x_out.max(), x_in.max())
                    x_range = np.linspace(x_min_full, x_max_full, 1000)
                    
                    # Interpolate both boundaries
                    # Use kind='linear' to preserve the horizontal flat segments at borders
                    print(x_out)
                    y_top_interp = interp1d(x_out, y_out, kind='linear', bounds_error=False, fill_value="extrapolate")(x_range)
                    y_bot_interp = interp1d(x_in, y_in, kind='linear', bounds_error=False, fill_value="extrapolate")(x_range)
                    
                    plt.fill_between(x_range, y_bot_interp, y_top_interp, 
                                    color='gray', alpha=0.4/num_layers, zorder=2)
                
                # # Draw the boundary lines
                # plt.plot(x_out, y_out, color='gray', linestyle=':', alpha=0.4, linewidth=1)
                # plt.plot(x_in, y_in, color='gray', linestyle=':', alpha=0.4, linewidth=1)

    if points: points = combine_path_collections(points)
    adjust_text(
        texts, 
        objects=points if points else None, 
        expand_text=(1.8, 2.2),
        expand_points=(1.5, 1.5),
        force_text=(1.5, 1.5),
        arrowprops=dict(arrowstyle='->', color='black', lw=0.5),
    )

    plt.xlabel(f"Precision (pass@{k1})", fontsize=16)
    plt.ylabel(f"Coverage (pass@{k2})", fontsize=16)
    plt.grid(True, linestyle="--", alpha=0.3)
    
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    by_label['$\\alpha$-DPG models'] = alpha_dpg_proxy
    by_label['Baseline methods'] = baseline_proxy
    plt.legend(by_label.values(), by_label.keys(), loc='best', fontsize=14, frameon=True)

    plt.tight_layout()
    os.makedirs("./figures", exist_ok=True)
    plt.savefig(f"./figures/frontier_plot.pdf", format='pdf', bbox_inches='tight')
    plt.show()

if __name__ == '__main__':
    from generate_plots_config import model_name_map
    
    models_to_plot = [
        # FILL IN THE MODEL NAMES
    ]

    result_paths = [
        # FILL IN THE MODEL PATH
    ]
    valid_paths = [p for p in result_paths if os.path.isdir(p)]
    
    if valid_paths:
        plot_convex_hull_comparison(
            models=models_to_plot, 
            model_name_map=model_name_map, 
            paths=valid_paths, 
            k_values=(1, 256),
            use_bootstrap=True, 
            show_errorbars=False
        )

In [ ]:
%load_ext autoreload
%autoreload 2
from generate_plots_config import model_name_map
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from matplotlib.colors import LinearSegmentedColormap

result_paths = [
        # FILL IN THE RESULT PATHS
]

def compute_difficulty_bin(rewards):
    """
    Categorize difficulty into bins based on fraction of correct samples:
    - 'easy': 50-100% correct
    - 'medium': 25-50% correct
    - 'hard': 0-10% correct
    - 'unsolved': no correct samples
    """
    rewards = np.array(rewards)
    if len(rewards) == 0 or np.sum(rewards) == 0:
        return 'unsolved'
    
    fraction_correct = np.mean(rewards)
    
    if fraction_correct >= 0.90:
        return 'easy'
    elif fraction_correct >= 0.3:
        return 'medium'
    elif fraction_correct > 0.0:
        return 'hard'
    else:
        return 'unsolved'

def load_results(model_name, result_paths):
    found_data = False
    for path in result_paths:
        filename = os.path.join(path, model_name, "results.parquet")
        if os.path.exists(filename):
            results  = pd.read_parquet(filename)
            return results
    if not found_data:
        print(f"ERROR: failed to find data for model {after_model}")
    return None

models = [
    # MODEL NAMES
]


for after_model in models:
    before_model = 'DeepSeek-Prover-V1.5-SFT'

    before_df = load_results(before_model, result_paths)
    after_df = load_results(after_model, result_paths)

    # Apply to dataframes
    before_bins = before_df['results'].apply(compute_difficulty_bin)
    after_bins  = after_df['results'].apply(compute_difficulty_bin)

    # Build a confusion matrix
    categories = ['easy', 'medium','hard', 'unsolved']
    matrix = pd.crosstab(
        before_bins, 
        after_bins, 
        rownames=['Base SFT'], 
        colnames=[f'{model_name_map.get(after_model, after_model)}'], 
        dropna=False
    ).reindex(index=categories, columns=categories, fill_value=0)

    # --- Start of Plotting Changes ---
    plt.figure(figsize=(7, 6))
    ax = plt.gca()

    # Define masks for the different parts of the matrix
    mask_lower = np.triu(np.ones_like(matrix, dtype=bool))
    mask_upper = np.tril(np.ones_like(matrix, dtype=bool))
    mask_diag = np.eye(matrix.shape[0], dtype=bool) == False
    
    # Define a consistent color scale range based on the entire matrix
    vmin = matrix.min().min()
    vmax = matrix.max().max()

    colors_blue = plt.cm.Blues(np.linspace(0, 1, 256))
    colors_blue[0] = (1, 1, 1, 1)  # RGBA for white
    cmap_blue_white_zero = LinearSegmentedColormap.from_list("blue_white_zero", colors_blue)

    # For reds
    colors_red = plt.cm.Reds(np.linspace(0, 1, 256))
    colors_red[0] = (1, 1, 1, 1)
    cmap_red_white_zero = LinearSegmentedColormap.from_list("red_white_zero", colors_red)

    # For greys
    colors_grey = plt.cm.Greys(np.linspace(0, 0.65, 256))
    colors_grey[0] = (1, 1, 1, 1)
    cmap_grey_white_zero = LinearSegmentedColormap.from_list("grey_white_zero", colors_grey)

    # Plot the lower triangle (blue)
    sns.heatmap(
        matrix, mask=mask_lower, cmap=cmap_blue_white_zero, ax=ax, cbar=False,
        annot=True, fmt="d", annot_kws={"size": 15}, vmin=vmin, vmax=vmax//3
    )
    # Plot the upper triangle (red)
    sns.heatmap(
        matrix, mask=mask_upper, cmap=cmap_red_white_zero, ax=ax, cbar=False,
        annot=True, fmt="d", annot_kws={"size": 15}, vmin=vmin, vmax=vmax//3
    )
    # Plot the diagonal (gray)
    sns.heatmap(
        matrix, mask=mask_diag, cmap=cmap_grey_white_zero, ax=ax, cbar=False,
        annot=True, fmt="d", annot_kws={"size": 15}, vmin=vmin, vmax=vmax
    )
    # --- End of Plotting Changes ---

    # Set labels and ticks
    ax.set_xlabel(f"{model_name_map.get(after_model, after_model)}", fontsize=14, labelpad=10)
    ax.set_ylabel("Base SFT", fontsize=14, labelpad=10)
    plt.yticks(rotation=0, fontsize=12)
    plt.xticks(rotation=0, fontsize=12)

    plt.tight_layout()
    
    # Create a directory for figures if it doesn't exist
    figure_dir = "./figures"
    os.makedirs(figure_dir, exist_ok=True)
    
    # Save as PDF
    save_path = os.path.join(figure_dir, f"difficulty_transition_matrix_{after_model}.pdf")
    plt.savefig(save_path, format='pdf')

    plt.show()

In [ ]:
%load_ext autoreload
%autoreload 2
from generate_plots_config import model_name_map
import pandas as pd
import math
import matplotlib.pyplot as plt
import os
import numpy as np
import json

def pass_at_k(n, c, k):
    """
    Calculates the pass@k metric.
    
    Args:
        n (int): Total number of samples.
        c (int): Number of correct samples.
        k (int): The 'k' in pass@k.
        
    Returns:
        float: The pass@k score.
    """
    if c == 0:
        return 0.0
    if n - c < k:
        return 1.0
    # Handle potential overflow with large numbers by calculating log combinations
    if n > 1000: # Heuristic threshold
        try:
            from scipy.special import comb
            return 1.0 - comb(n - c, k, exact=False) / comb(n, k, exact=False)
        except ImportError:
            pass # Fallback to math.comb if scipy is not available
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)

def load_results(model_name, result_paths):
    found_data = False
    for path in result_paths:
        filename = os.path.join(path, model_name, "results.parquet")
        if os.path.exists(filename):
            results  = pd.read_parquet(filename)
            return results
    if not found_data:
        print(f"ERROR: failed to find data for model {model_name}")
    return results

def plot_pass_at_k(models, model_name_map, paths, ks, n_samples_json=128):
    """
    Generates a plot of pass@k vs. k for a list of models, with a fallback data loading strategy.
    
    Args:
        models (list): List of model directory names.
        model_name_map (dict): Dictionary to map model names to display names.
        paths (list or str): List of base paths to search for model data.
        ks (list): List of k values to evaluate.
        n_samples_json (int): Constant number of samples to assume for JSON data.
    """
    if not isinstance(paths, list):
        paths = [paths]

    plt.figure(figsize=(8, 6))


    # --- Data Loading and Plotting Loop ---
    print("Searching for result files and calculating pass@k...")
    for model in models:
        passk_values = None
        found_data = False
        ds = load_results(model, result_paths)
        if ds is not None:
            passk_values = []
            for n in ks:
                pass_k_val = ds['results'].apply(lambda x: pass_at_k(len(x), int(sum(x)), n)).mean()
                passk_values.append(pass_k_val)
        else:
            print(f"Warning: No results file found for model '{model}' in any provided path. Skipping.")
            continue
        
        plt.plot(
            ks, 
            passk_values, 
            marker=model_marker_map.get(model, 'o'), 
            color=model_color_map.get(model, 'black'), 
            linestyle='--', 
            label=model_name_map.get(model, model),
            markersize=8 
        )

    # --- Plot Styling and Saving ---
    plt.xscale("log", base=2)
    plt.xticks(ks, labels=[f"{k}" for k in ks])
    plt.xlabel("k", fontsize=15)
    plt.ylabel("pass@k", fontsize=15)
    plt.title("Pass@k for α-DPG variants", fontsize=15)
    plt.grid(True, linestyle="--", alpha=0.6)
    
    # --- Sort Legend ---
    handles, labels = plt.gca().get_legend_handles_labels()
    sorted_legend_items = sorted(zip(labels, handles), key=lambda x: x[0])
    if sorted_legend_items:
        sorted_labels, sorted_handles = zip(*sorted_legend_items)
        plt.legend(sorted_handles, sorted_labels, title="Models", fontsize=12)

    plt.tight_layout()
    
    figure_dir = "./figures"
    os.makedirs(figure_dir, exist_ok=True)
    save_path = os.path.join(figure_dir, "alpha_pass_at_k_plot.pdf")
    plt.savefig(save_path, format='pdf')
    
    print(f"\nPlot saved to {save_path}")
    plt.show()

# --- Main execution block ---
if __name__ == '__main__':
    models_to_plot = [
        # MODEL NAMES
    ]
    

    result_paths = [
        # RESULT PATH
    ]
    
    ks_to_evaluate = [2**k for k in range(9)] # [1, 2, 4, ..., 256]

    valid_paths = [p for p in result_paths if os.path.isdir(p)]
    if not valid_paths:
        print(f"Error: None of the specified base paths exist: {result_paths}")
    else:
        plot_pass_at_k(
            models=models_to_plot, 
            model_name_map=model_name_map, 
            paths=valid_paths, 
            ks=ks_to_evaluate,
            n_samples_json=128
        )


In [ ]:
import pandas as pd
import math
import os
import numpy as np
import json
from scipy.special import comb

# --- Helper Functions (Same as before) ---

def pass_at_k(n, c, k):
    if c == 0:
        return 0.0
    if n - c < k:
        return 1.0
    if n > 1000: 
        try:
            return 1.0 - comb(n - c, k, exact=False) / comb(n, k, exact=False)
        except ImportError:
            pass 
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)

def load_model_data(model_name, paths, n_samples_json=128):
    for path in paths:
        # Check Parquet
        for parquet_file in [os.path.join(path, model_name, "results.parquet"), 
                             os.path.join(path, model_name, "merged_results.parquet"), 
                             os.path.join(path, model_name + '.eval.parquet')]:
            if os.path.exists(parquet_file):
                ds = pd.read_parquet(parquet_file)
                return 'parquet', ds['results']

        # Check JSON
        json_file = os.path.join(path, f"{model_name}.eval.json")
        if os.path.exists(json_file):
            with open(json_file, 'r') as f:
                data = json.load(f)
            if data and isinstance(list(data.values())[0], list):
                return 'json', np.array(list(data.values())[0])
    
    print(f"Warning: Data not found for {model_name}")
    return None, None

def get_bootstrap_distribution(data_type, data, k, bootstrap_indices, n_samples_json=128):
    if data_type == 'parquet':
        data_arr = data.values
        per_problem_scores = np.array([pass_at_k(len(x), int(sum(x)), k) for x in data_arr])
        resampled_scores = per_problem_scores[bootstrap_indices]
        return np.mean(resampled_scores, axis=1)
    elif data_type == 'json':
        per_problem_scores = np.array([
            pass_at_k(n_samples_json, int(round(acc * n_samples_json)), k) 
            for acc in data
        ])
        resampled_scores = per_problem_scores[bootstrap_indices]
        return np.mean(resampled_scores, axis=1)
    return np.array([])

def calculate_p_value(dist_a, dist_b):
    diffs = dist_a - dist_b
    observed_diff = np.mean(diffs)
    if observed_diff > 0:
        p_val = 2 * np.mean(diffs <= 0)
    else:
        p_val = 2 * np.mean(diffs >= 0)
    return max(0.0, min(1.0, p_val)), observed_diff

def escape_latex(s):
    return s.replace("_", r"\_").replace("%", r"\%")

# --- Matrix Generation Logic ---

def print_latex_matrix(models, model_map, distributions, k_list):
    """
    Generates a LaTeX matrix where:
    - Diagonal: Absolute Score (Pass@1 / Pass@256)
    - Upper Triangle: Pass@256 Difference (Row - Col)
    - Lower Triangle: Pass@1 Difference (Row - Col)
    """
    n = len(models)
    k1, k2 = k_list[0], k_list[1] # k1=1, k2=256
    
    print("\n" + "="*20 + " LaTeX Matrix Table " + "="*20)
    print(r"\begin{table*}[h]")
    print(r"\centering")
    print(r"\scriptsize") # Small font to fit many columns
    print(r"\setlength{\tabcolsep}{3pt}") # Tight columns
    
    # Dynamic column definition
    print(r"\begin{tabular}{l" + "c" * n + "}")
    print(r"\toprule")
    
    # --- Header Row (Vertical Names) ---
    header = " & "
    for m in models:
        display_name = model_map.get(m, m)
        header += f"\\rotatebox{{90}}{{{escape_latex(display_name)}}} & "
    header = header[:-2] + r"\\"
    print(header)
    print(r"\midrule")
    
    # --- Matrix Rows ---
    for i, row_model in enumerate(models):
        row_label = model_map.get(row_model, row_model)
        row_str = f"{escape_latex(row_label)} & "
        
        for j, col_model in enumerate(models):
            
            # -- Diagonal: Absolute Scores --
            if i == j:
                abs_k1 = np.mean(distributions[(row_model, k1)]) * 100
                abs_k2 = np.mean(distributions[(row_model, k2)]) * 100
                # Format: 12.5 / 45.2
                row_str += f"\\textbf{{{abs_k1:.1f}/{abs_k2:.1f}}} & "
            
            # -- Upper Triangle: Pass@256 (k2) Differences --
            elif j > i:
                # Calculate Row - Col
                dist_row = distributions[(row_model, k2)]
                dist_col = distributions[(col_model, k2)]
                p_val, diff = calculate_p_value(dist_row, dist_col)
                
                diff_pct = diff * 100
                cell_text = f"{diff_pct:+.1f}"
                
                # Bold if significant
                if p_val < 0.05:
                    cell_text = f"\\textbf{{{cell_text}}}"
                
                # Optional: Gray out insignificant results for clarity
                # else: cell_text = f"\\textcolor{{gray}}{{{cell_text}}}"

                row_str += f"{cell_text} & "

            # -- Lower Triangle: Pass@1 (k1) Differences --
            elif j < i:
                # Calculate Row - Col
                dist_row = distributions[(row_model, k1)]
                dist_col = distributions[(col_model, k1)]
                p_val, diff = calculate_p_value(dist_row, dist_col)
                
                diff_pct = diff * 100
                cell_text = f"{diff_pct:+.1f}"
                
                if p_val < 0.05:
                    cell_text = f"\\textbf{{{cell_text}}}"
                
                row_str += f"{cell_text} & "
        
        # Cleanup row end
        row_str = row_str[:-2] + r"\\"
        print(row_str)

    print(r"\bottomrule")
    print(r"\end{tabular}")
    print(r"\caption{Pairwise performance comparison. \textbf{Diagonal}: Absolute Pass@1 / Pass@256 scores (\%). \textbf{Upper Triangle}: Pass@256 differences (Row $-$ Column). \textbf{Lower Triangle}: Pass@1 differences (Row $-$ Column). \textbf{Bold} indicates statistical significance ($p < 0.05$) via paired bootstrap.}")
    print(r"\label{tab:pairwise_matrix}")
    print(r"\end{table*}")
    print("="*60)

# --- Main Execution ---
if __name__ == "__main__":
    
    # --- User Specified Inputs ---
    models_to_process = [
        # MODEL NAMES
    ]

    result_paths = [
        # RESULT PATH
    ]


    # --- Processing ---
    
    # 1. Filter out commented lines and ensure valid paths
    valid_models = [m for m in models_to_process if m and not m.startswith('#')]
    valid_paths = [p for p in result_paths if os.path.isdir(p)]
    
    if not valid_paths:
        print("Error: No valid directories found.")
        exit()

    # 2. Load Data
    loaded_data = {}
    n_problems = None
    print(f"Loading data for {len(valid_models)} models...")
    
    final_model_list = []
    
    for model in valid_models:
        dtype, dvals = load_model_data(model, valid_paths)
        if dtype:
            # Ensure sample alignment
            if n_problems is None: n_problems = len(dvals)
            
            if len(dvals) != n_problems:
                print(f"  Skipping {model}: length mismatch (got {len(dvals)}, expected {n_problems})")
                continue
                
            loaded_data[model] = (dtype, dvals)
            final_model_list.append(model)

    # 3. Bootstrap
    n_resamples = 10000
    k_list = [1, 256]
    
    print(f"Bootstrapping {n_resamples} times...")
    bootstrap_indices = np.random.randint(0, n_problems, size=(n_resamples, n_problems))
    distributions = {}
    
    for model in final_model_list:
        dtype, dvals = loaded_data[model]
        for k in k_list:
            dist = get_bootstrap_distribution(dtype, dvals, k, bootstrap_indices)
            distributions[(model, k)] = dist

    # 4. Print Matrix
    if final_model_list:
        print_latex_matrix(final_model_list, model_name_map, distributions, k_list)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from adjustText import adjust_text
import numpy as np
import re
from collections import Counter
import math
import pandas as pd

def get_count(arr):
    tactics = []
    premises = []

    for proof in arr:
        premises_matches = re.findall(r'\[([^\]]+)\]', proof)
        for pm in premises_matches:
            for p in re.split(r'[\s,]+', pm):
                if p:
                    premises.append(p.strip())

        lines = proof.splitlines()
        for line in lines:
            line = line.strip().strip("`")
            if not line:
                continue
            first = line.split()[0]
            tactics.append(first)

    tactic_freqs = Counter(tactics)
    premise_freqs = Counter(premises)
    return tactic_freqs, premise_freqs

def get_metrics(ds):
    list_tactic_freqs=[]
    list_premise_freqs=[]
    list_shannon_entropy_tactic=[]
    list_shannon_entropy_premise=[]
    list_simpson_entropy_tactic=[]
    list_simpson_entropy_premise=[]

    # Functions for diversity metrics
    def shannon_entropy(counter):
        total = sum(counter.values())
        if total == 0:
            return 0.0
        return -sum((count/total) * math.log(count/total, 2) for count in counter.values())

    def simpson_index(counter):
        total = sum(counter.values())
        if total == 0:
            return 0.0
        return 1 - sum((count/total)**2 for count in counter.values())


    for i in range(len(ds)):

        
        arr = ds['responses'][i]

        tactic_freqs, premise_freqs = get_count(arr)
 
        list_tactic_freqs.append(tactic_freqs)
        list_premise_freqs.append(premise_freqs)
        list_shannon_entropy_tactic.append(shannon_entropy(tactic_freqs))
        list_shannon_entropy_premise.append(shannon_entropy(premise_freqs))
        list_simpson_entropy_tactic.append(simpson_index(tactic_freqs))
        list_simpson_entropy_premise.append(simpson_index(premise_freqs))

    return {
        "Premise_Simpson_Index": np.mean(list_simpson_entropy_premise),
        "Tactic_Simpson_Index": np.mean(list_simpson_entropy_tactic),
        "Tactic_Entropy_Index": np.mean(list_shannon_entropy_tactic),
        "Premise_Entropy_Index": np.mean(list_shannon_entropy_premise),
    }


models = [
        # MODEL NAMES
    ]

# build dataframe
from tqdm import tqdm 
results = []
results_pass = []
results_pass1=[]
for model in tqdm(models):
    ds = None
    for path in [
        # RESULT PATH
    ]:
        if os.path.exists(path):
            ds = pd.read_parquet(path)
    if ds is None:
        print(f"Result not found for {model}")
        continue
    metrics = get_metrics(ds)
    results.append({
        "Model": model_name_map[model],  # pretty name
        **metrics
    })
    results_pass.append(ds['results'].apply(lambda x: pass_at_k(len(x), int(sum(x)), 256)).mean())
    results_pass1.append(ds['results'].apply(lambda x: pass_at_k(len(x), int(sum(x)), 1)).mean())




    

df = pd.DataFrame(results)
df['pass@256'] = results_pass
df['pass@1'] =results_pass1
#df.set_index("Model", inplace=True)

print(df.to_latex())
def pass_at_k(n, c, k):
    """
    Calculates the pass@k metric.
    n: total number of samples.
    c: number of correct samples.
    k: the 'k' in pass@k.
    """
    if c == 0:
        return 0.0
    if n - c < k:
        return 1.0
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)



def combine_path_collections(
    pathcollections: list[PathCollection],
) -> PathCollection:
    """
    Correctly combines multiple PathCollection objects into a single one.

    This function gathers not only paths and sizes but also the crucial
    offsets (coordinates) and colors.
    """
    # Initialize lists to store all properties
    all_paths = []
    all_sizes = []
    all_offsets = []
    all_facecolors = []
    all_edgecolors = []

    # Loop through each collection and gather its properties
    for pc in pathcollections:
        # It's possible a collection is empty, so we check
        if pc.get_paths():
            all_paths.extend(pc.get_paths() * len(pc.get_offsets()))
            all_sizes.extend(pc.get_sizes())
            all_offsets.extend(pc.get_offsets())
            all_facecolors.extend(pc.get_facecolors())
            all_edgecolors.extend(pc.get_edgecolors())

    # Create the new collection with the visual properties
    combined_pc = PathCollection(
        all_paths,
        sizes=all_sizes,
        facecolors=all_facecolors,
        edgecolors=all_edgecolors,
    )

    # **Crucially, set the offsets on the new collection**
    combined_pc.set_offsets(np.array(all_offsets))

    return combined_pc

def diverisity_plot(diversity_index, object_kind, diversity_kind):
    # Create the figure and axes
    fig, ax1 = plt.subplots(figsize=(10, 8))

    # Define colors
    color_pass1 = "#1f77b4"
    color_pass256 = "#d62728"

    # --- Filter only 'amari' models for regression ---
    df_amari = df[df["Model"].str.contains("DPG", case=False)]

    # --- Polynomial Regression (Order 2) Calculation ---
    # pass@1 regression
    x_pass1 = df_amari[diversity_index]
    y_pass1 = df_amari["pass@1"]
    coeffs1 = np.polyfit(x_pass1, y_pass1, 1)
    poly1 = np.poly1d(coeffs1)

    # pass@256 regression
    x_pass256 = df_amari[diversity_index]
    y_pass256 = df_amari["pass@256"]
    coeffs2 = np.polyfit(x_pass256, y_pass256, 1)
    poly2 = np.poly1d(coeffs2)

    # Generate a range of x values for the regression lines
    x_range = np.linspace(df[diversity_index].min(),
                        df[diversity_index].max(), 100)

    # --- Plotting ---
    points_1, points_2 = [], []
    texts_1, texts_2 = [], []
    ax1.set_xlabel(f"{object_kind} {diversity_kind} Index", fontsize=12)
    ax1.set_ylabel("Pass@1", color=color_pass1, fontsize=12)
    points_1.append(ax1.scatter(df[diversity_index], df["pass@1"],
                color=color_pass1, s=100, label="pass@1", alpha=0.8, zorder=2))
    ax1.plot(x_range, poly1(x_range), color=color_pass1,
            linestyle='--', linewidth=2, label="Linear Regression (α-DPG)")
    ax1.tick_params(axis="y", labelcolor=color_pass1)

    ax2 = ax1.twinx()
    ax2.set_ylabel("Pass@256", color=color_pass256, fontsize=12)
    points_2.append(ax2.scatter(df[diversity_index], df["pass@256"],
                color=color_pass256, s=100, marker="D", label="pass@256", alpha=0.8, zorder=2))
    ax2.plot(x_range, poly2(x_range), color=color_pass256,
            linestyle=':', linewidth=2, label="Linear Regression (α-DPG)")
    ax2.tick_params(axis="y", labelcolor=color_pass256)

    # Title and legend
    fig.suptitle(f"{object_kind} Diversity ({diversity_kind}) vs Model Pass@k", fontsize=16, fontweight="bold")
    ax1.grid(True, linestyle="--", alpha=0.5)

    # Create a combined legend for all elements
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2, loc='center left', fontsize=10)

    # Store the positions of placed labels to avoid overlaps
    placed_positions = []

    def is_overlapping(x, y, placed_positions, threshold=0.02):
        """Check if a new label position overlaps with existing ones."""
        for px, py in placed_positions:
            distance = ((x - px)**2 + (y - py)**2)**0.5
            if distance < threshold:
                return True
        return False

    # Iterate and place labels for pass@1
    for i, row in df.iterrows():
        x, y = row[diversity_index], row["pass@1"]
        offset = -0.01
        while is_overlapping(x + offset, y, placed_positions):
            offset -= 0.005
        texts_1.append(ax1.text(x + offset, y, row["Model"], fontsize=8, color=color_pass1, ha='left'))
        placed_positions.append((x + offset, y))

    # Iterate and place labels for pass@256
    for i, row in df.iterrows():
        x, y = row[diversity_index], row["pass@256"]
        offset = -0.01
        # while is_overlapping(x + offset, y, placed_positions):
        #     offset -= 0.005
        texts_2.append(ax2.text(x + offset, y, row["Model"], fontsize=8, color=color_pass256, ha='right'))
        placed_positions.append((x + offset, y))

    points_1 = combine_path_collections(points_1)
    points_2 = combine_path_collections(points_2)
    adjust_text(texts_1, ax=ax1, objects=points_1, expand=(1, 1.5))
    adjust_text(texts_2, ax=ax2, objects=points_2, expand=(1, 1.5))
    # Final plot adjustments
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])

diverisity_plot("Premise_Simpson_Index", "Premise", "Simpson Index")
plt.savefig("diversity_vs_performance_quadratic_amari_si.pdf")
diverisity_plot("Premise_Entropy_Index", "Premise", "Shannon Entropy")
plt.savefig("diversity_vs_performance_quadratic_amari_e.pdf")
diverisity_plot("Tactic_Simpson_Index", "Tactic", "Simpson Index")
plt.savefig("diversity_vs_performance_quadratic_amari_tact_si.pdf")
diverisity_plot("Tactic_Entropy_Index", "Tactic", "Shannon Entropy")
plt.savefig("diversity_vs_performance_quadratic_amari_tact_e.pdf")
